In [ ]:
# parte 2 vamos analisaer um único debate  - 

In [ ]:
# imports 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA

In [ ]:
# abrir o debate escolhido - Gouveia Melo vs Cotrim Figueiredo

DATA_DIR = Path(".")

audio_files = sorted(DATA_DIR.glob("*_audio.pkl"))

for f in audio_files:
    if "Gouveia_Melo" in f.name and "Cotrim" in f.name and "November_20" in f.name:
        debate_file = f
        break

print("Ficheiro escolhido:", debate_file.name)

df_debate = pd.read_pickle(debate_file)

print("Dimensão:", df_debate.shape)
df_debate.head()

In [ ]:
df_debate = df_debate.copy()

df_debate["segment_start"] = df_debate["time stamp"]
df_debate["segment_end"] = df_debate["time stamp"] + df_debate["duration"]

df_debate[["segment_start", "duration", "segment_end"]].head()

In [ ]:
# agrupar as 3 vozes em embedding 

embeddings = np.vstack(df_debate["speak_embeddings"].values)

embeddings_norm = normalize(embeddings)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_debate["speaker_cluster"] = kmeans.fit_predict(embeddings_norm)

df_debate[["segment_start", "duration", "speaker_cluster"]].head()

In [ ]:
speaker_time = (
    df_debate
    .groupby("speaker_cluster")
    .agg(
        total_speech_sec=("duration", "sum"),
        n_segments=("duration", "count"),
        mean_segment_duration=("duration", "mean"),
        mean_speechrate=("speechrate", "mean")
    )
    .reset_index()
    .sort_values("total_speech_sec", ascending=False)
)

speaker_time["person_label"] = [
    f"Person {i}" for i in range(1, len(speaker_time) + 1)
]

cluster_to_person = dict(
    zip(speaker_time["speaker_cluster"], speaker_time["person_label"])
)

df_debate["person"] = df_debate["speaker_cluster"].map(cluster_to_person)

speaker_time["total_speech_min"] = speaker_time["total_speech_sec"] / 60
speaker_time["speech_share"] = (
    speaker_time["total_speech_sec"] / speaker_time["total_speech_sec"].sum()
)

speaker_time = speaker_time.round(3)

speaker_time

In [ ]:
plot_data = speaker_time.sort_values("speech_share", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(plot_data["person_label"], plot_data["speech_share"])

plt.title("Percentagem de tempo de fala por pessoa")
plt.xlabel("Percentagem do tempo total de fala")
plt.ylabel("Pessoa")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.decomposition import PCA

embeddings = np.vstack(df_debate["speak_embeddings"].values)
embeddings_norm = normalize(embeddings)

pca = PCA(n_components=2)
pca_result = pca.fit_transform(embeddings_norm)

df_debate["PCA1"] = pca_result[:, 0]
df_debate["PCA2"] = pca_result[:, 1]

plt.figure(figsize=(8, 6))

for person in ["Person 1", "Person 2", "Person 3"]:
    data = df_debate[df_debate["person"] == person]
    plt.scatter(data["PCA1"], data["PCA2"], label=person, alpha=0.7)

plt.title("PCA dos speaker embeddings")
plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))

colors = {
    "Person 1": "tab:blue",
    "Person 2": "tab:orange",
    "Person 3": "tab:green"
}

for person in ["Person 1", "Person 2", "Person 3"]:
    data = df_debate[df_debate["person"] == person]
    
    plt.scatter(
        data["PCA1"],
        data["PCA2"],
        label=person,
        alpha=0.7,
        s=45,
        color=colors[person]
    )
    
    # centro do grupo
    center_x = data["PCA1"].mean()
    center_y = data["PCA2"].mean()
    
    plt.scatter(
        center_x,
        center_y,
        color=colors[person],
        s=180,
        marker="X",
        edgecolor="black"
    )
    
    # zona aproximada do grupo
    radius_x = data["PCA1"].std() * 2
    radius_y = data["PCA2"].std() * 2
    
    ellipse = plt.matplotlib.patches.Ellipse(
        (center_x, center_y),
        width=radius_x * 2,
        height=radius_y * 2,
        alpha=0.15,
        color=colors[person]
    )
    
    plt.gca().add_patch(ellipse)

plt.title("PCA dos speaker embeddings com zonas por pessoa")
plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.legend(title="Speaker")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Timeline dos segmentos de fala por pessoa

person_order = ["Person 1", "Person 2", "Person 3"]

plt.figure(figsize=(14, 4))

for i, person in enumerate(person_order):
    person_segments = df_debate[df_debate["person"] == person]
    
    for _, row in person_segments.iterrows():
        plt.broken_barh(
            [(row["segment_start"], row["duration"])],
            (i - 0.4, 0.8)
        )

plt.yticks(range(len(person_order)), person_order)
plt.xlabel("Tempo do debate (segundos)")
plt.ylabel("Pessoa")
plt.title("Timeline dos segmentos de fala por pessoa")
plt.tight_layout()
plt.show()

# cada candidato falou em torno de 14 minutos e pouco

In [ ]:
speaker_diagnostics = (
    df_debate
    .groupby("person")
    .agg(
        n_segments=("duration", "count"),
        total_speech_sec=("duration", "sum"),
        mean_duration=("duration", "mean"),
        median_duration=("duration", "median"),
        max_duration=("duration", "max"),
        first_time=("segment_start", "min"),
        last_time=("segment_end", "max"),
        mean_speechrate=("speechrate", "mean")
    )
    .reset_index()
)

speaker_diagnostics["total_speech_min"] = speaker_diagnostics["total_speech_sec"] / 60
speaker_diagnostics["speech_share"] = speaker_diagnostics["total_speech_sec"] / speaker_diagnostics["total_speech_sec"].sum()

speaker_diagnostics = speaker_diagnostics.round(3)

speaker_diagnostics.sort_values("total_speech_sec", ascending=False)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path(".")

debate_file = Path("Cotrim_Figueiredo_vs_Gouveia_Melo_November_20_audio.pkl")

print("Existe?", debate_file.exists())

df_debate = pd.read_pickle(debate_file)

df_debate = df_debate.copy()
df_debate["segment_start"] = df_debate["time stamp"]
df_debate["segment_end"] = df_debate["time stamp"] + df_debate["duration"]
df_debate = df_debate.sort_values("segment_start").reset_index(drop=True)
df_debate["segment_id"] = df_debate.index

print(df_debate.shape)
df_debate[["segment_id", "segment_start", "segment_end", "duration"]].head()

In [ ]:
# PARTE: identificação automática de vozes usando os 28 debates

from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA

DATA_DIR = Path(".")
audio_files = sorted(DATA_DIR.glob("*_audio.pkl"))

def extract_metadata_from_filename(file_path):
    filename = file_path.name.replace("_audio.pkl", "")
    candidate_part, date_part = filename.split("_vs_")
    
    candidate_1 = candidate_part.replace("_", " ")
    
    date_parts = date_part.split("_")
    month = date_parts[-2]
    day = int(date_parts[-1])
    candidate_2 = " ".join(date_parts[:-2])
    
    debate_name = f"{candidate_1} vs {candidate_2}"
    
    return {
        "source_file": file_path.name,
        "debate_name": debate_name,
        "candidate_1": candidate_1,
        "candidate_2": candidate_2,
        "month": month,
        "day": day
    }

all_dfs = []

for file_path in audio_files:
    df = pd.read_pickle(file_path)
    metadata = extract_metadata_from_filename(file_path)
    
    for key, value in metadata.items():
        df[key] = value
    
    all_dfs.append(df)

audio_all = pd.concat(all_dfs, ignore_index=True)

audio_all["segment_start"] = audio_all["time stamp"]
audio_all["segment_end"] = audio_all["time stamp"] + audio_all["duration"]

print(audio_all.shape)
audio_all[["debate_name", "candidate_1", "candidate_2"]].drop_duplicates().head()

In [ ]:
# criar clusters globais de vozes

embeddings = np.vstack(audio_all["speak_embeddings"].values)
embeddings_norm = normalize(embeddings)

unique_candidates = sorted(
    set(audio_all["candidate_1"].unique()).union(set(audio_all["candidate_2"].unique()))
)

print("Candidatos encontrados:", unique_candidates)

# número de clusters: candidatos + possíveis moderadores/ruído
n_global_clusters = len(unique_candidates) + 5

kmeans_global = KMeans(n_clusters=n_global_clusters, random_state=42, n_init=10)
audio_all["global_voice_cluster"] = kmeans_global.fit_predict(embeddings_norm)

audio_all["global_voice_cluster"].value_counts().sort_index()

In [ ]:
# associar clusters globais a candidatos

candidate_scores = []

for candidate in unique_candidates:
    candidate_debates = audio_all[
        (audio_all["candidate_1"] == candidate) |
        (audio_all["candidate_2"] == candidate)
    ]["debate_name"].unique()
    
    for cluster in sorted(audio_all["global_voice_cluster"].unique()):
        cluster_df = audio_all[audio_all["global_voice_cluster"] == cluster]
        
        total_cluster_duration = cluster_df["duration"].sum()
        
        duration_in_candidate_debates = cluster_df[
            cluster_df["debate_name"].isin(candidate_debates)
        ]["duration"].sum()
        
        debates_where_cluster_appears = cluster_df[
            cluster_df["debate_name"].isin(candidate_debates)
        ].groupby("debate_name")["duration"].sum()
        
        coverage = (debates_where_cluster_appears > 10).sum() / len(candidate_debates)
        
        share = duration_in_candidate_debates / total_cluster_duration
        
        score = share * coverage
        
        candidate_scores.append({
            "candidate": candidate,
            "cluster": cluster,
            "score": score,
            "share_in_candidate_debates": share,
            "coverage": coverage
        })

candidate_scores = pd.DataFrame(candidate_scores).round(3)

candidate_scores.sort_values("score", ascending=False).head(20)

In [ ]:
# escolher automaticamente o melhor cluster para cada candidato

assigned_candidates = {}
used_clusters = set()

scores_sorted = candidate_scores.sort_values("score", ascending=False)

for _, row in scores_sorted.iterrows():
    candidate = row["candidate"]
    cluster = row["cluster"]
    
    if candidate not in assigned_candidates and cluster not in used_clusters:
        assigned_candidates[candidate] = {
            "cluster": cluster,
            "score": row["score"],
            "share": row["share_in_candidate_debates"],
            "coverage": row["coverage"]
        }
        used_clusters.add(cluster)

candidate_cluster_map = pd.DataFrame([
    {
        "candidate": candidate,
        "cluster": values["cluster"],
        "score": values["score"],
        "share": values["share"],
        "coverage": values["coverage"]
    }
    for candidate, values in assigned_candidates.items()
]).sort_values("candidate")

candidate_cluster_map

In [ ]:
# escolher o debate Gouveia Melo vs Cotrim Figueiredo

debate_mask = (
    audio_all["source_file"].str.contains("Gouveia_Melo") &
    audio_all["source_file"].str.contains("Cotrim") &
    audio_all["source_file"].str.contains("November_20")
)

df_debate_auto = audio_all[debate_mask].copy()

print(df_debate_auto["source_file"].unique())
print(df_debate_auto.shape)

candidate_1 = df_debate_auto["candidate_1"].iloc[0]
candidate_2 = df_debate_auto["candidate_2"].iloc[0]

print(candidate_1, "vs", candidate_2)

In [ ]:
# mapear clusters para nomes prováveis

cluster_to_candidate = dict(
    zip(candidate_cluster_map["cluster"], candidate_cluster_map["candidate"])
)

df_debate_auto["predicted_person"] = df_debate_auto["global_voice_cluster"].map(cluster_to_candidate)

# no debate escolhido, só queremos os dois candidatos; o resto é moderador/outro
valid_people = [candidate_1, candidate_2]

df_debate_auto["predicted_person"] = df_debate_auto["predicted_person"].where(
    df_debate_auto["predicted_person"].isin(valid_people),
    "Moderator/Other"
)

df_debate_auto[[
    "segment_start",
    "duration",
    "segment_end",
    "global_voice_cluster",
    "predicted_person"
]].head(20)

In [ ]:
person_time_auto = (
    df_debate_auto
    .groupby("predicted_person")
    .agg(
        n_segments=("duration", "count"),
        total_speech_sec=("duration", "sum"),
        mean_duration=("duration", "mean"),
        median_duration=("duration", "median"),
        max_duration=("duration", "max"),
        mean_speechrate=("speechrate", "mean")
    )
    .reset_index()
)

person_time_auto["total_speech_min"] = person_time_auto["total_speech_sec"] / 60
person_time_auto["speech_share"] = (
    person_time_auto["total_speech_sec"] / person_time_auto["total_speech_sec"].sum()
)

person_time_auto = person_time_auto.round(3)

person_time_auto.sort_values("total_speech_sec", ascending=False)

In [ ]:
person_order = [candidate_1, candidate_2, "Moderator/Other"]

plt.figure(figsize=(14, 4))

for i, person in enumerate(person_order):
    person_segments = df_debate_auto[df_debate_auto["predicted_person"] == person]
    
    for _, row in person_segments.iterrows():
        plt.broken_barh(
            [(row["segment_start"], row["duration"])],
            (i - 0.4, 0.8)
        )

plt.yticks(range(len(person_order)), person_order)
plt.xlabel("Tempo do debate (segundos)")
plt.ylabel("Pessoa")
plt.title("Timeline dos segmentos de fala estimados por pessoa")
plt.tight_layout()
plt.show()

In [ ]:
# ver quantos clusters ficaram em cada pessoa

pd.crosstab(
    df_debate_auto["global_voice_cluster"],
    df_debate_auto["predicted_person"]
)

In [ ]:
person_time_auto = (
    df_debate_auto
    .groupby("predicted_person")
    .agg(
        n_segments=("duration", "count"),
        total_speech_sec=("duration", "sum"),
        mean_duration=("duration", "mean"),
        median_duration=("duration", "median"),
        max_duration=("duration", "max"),
        mean_speechrate=("speechrate", "mean")
    )
    .reset_index()
)

person_time_auto["total_speech_min"] = person_time_auto["total_speech_sec"] / 60

person_time_auto["speech_share"] = (
    person_time_auto["total_speech_sec"] /
    person_time_auto["total_speech_sec"].sum()
)

person_time_auto = person_time_auto.round(3)

person_time_auto.sort_values("total_speech_min", ascending=False)

In [ ]:
print("Total de fala segmentada no ficheiro:")
print(round(person_time_auto["total_speech_min"].sum(), 2), "minutos")

In [ ]:
# gráfico do tempo que cada um fala

plot_data = person_time_auto.sort_values("total_speech_min", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(plot_data["predicted_person"], plot_data["total_speech_min"])

plt.title("Tempo de fala por participante")
plt.xlabel("Tempo de fala (minutos)")
plt.ylabel("Participante")
plt.tight_layout()
plt.show()

# agora vamos ver os debates que deteta melhor o audio ao longo do tempo, dai correr todos e ver se o tempo que cada pessoa fala coincide com o tempo real que eles falaram.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# pasta para guardar os gráficos
output_dir = Path("speaker_time_graphs")
output_dir.mkdir(exist_ok=True)

all_speaker_summaries = []

for file_path in audio_files:
    # abrir ficheiro
    df = pd.read_pickle(file_path).copy()
    
    # nome do debate
    debate_name = file_path.name.replace("_audio.pkl", "").replace("_", " ")
    
    # criar tempos
    df["segment_start"] = df["time stamp"]
    df["segment_end"] = df["time stamp"] + df["duration"]
    
    # embeddings
    embeddings = np.vstack(df["speak_embeddings"].values)
    embeddings_norm = normalize(embeddings)
    
    # clustering em 3 vozes
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df["speaker_cluster"] = kmeans.fit_predict(embeddings_norm)
    
    # tempo por cluster
    speaker_time = (
        df
        .groupby("speaker_cluster")
        .agg(
            n_segments=("duration", "count"),
            total_speech_sec=("duration", "sum"),
            mean_duration=("duration", "mean"),
            mean_speechrate=("speechrate", "mean")
        )
        .reset_index()
        .sort_values("total_speech_sec", ascending=False)
    )
    
    # ordenar como Person 1, Person 2, Person 3
    speaker_time["person"] = [f"Person {i}" for i in range(1, len(speaker_time) + 1)]
    speaker_time["total_speech_min"] = speaker_time["total_speech_sec"] / 60
    speaker_time["speech_share"] = (
        speaker_time["total_speech_sec"] / speaker_time["total_speech_sec"].sum()
    )
    
    speaker_time["debate_name"] = debate_name
    all_speaker_summaries.append(speaker_time)
    
    # gráfico
    plot_data = speaker_time.sort_values("total_speech_min", ascending=True)
    
    plt.figure(figsize=(8, 5))
    bars = plt.barh(plot_data["person"], plot_data["total_speech_min"])
    
    for bar in bars:
        width = bar.get_width()
        plt.text(
            width + 0.1,
            bar.get_y() + bar.get_height()/2,
            f"{width:.2f} min",
            va="center"
        )
    
    plt.title(f"Tempo de fala estimado por speaker\n{debate_name}")
    plt.xlabel("Tempo de fala estimado (minutos)")
    plt.ylabel("Speaker estimado")
    plt.tight_layout()
    
    # guardar gráfico
    safe_name = file_path.stem.replace("_audio", "")
    plt.savefig(output_dir / f"{safe_name}_speaker_time.png", dpi=300, bbox_inches="tight")
    plt.show()

# agora verificámos que há debates que são mais audiveis em termos de audio, por dicção e acabamos por escolher 0 ventura vs seguro 


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from matplotlib.patches import Ellipse

DATA_DIR = Path(".")

audio_files = sorted(DATA_DIR.glob("*_audio.pkl"))

debate_file = None

for f in audio_files:
    if "Ventura" in f.name and "Seguro" in f.name and "November_17" in f.name:
        debate_file = f
        break

print("Ficheiro escolhido:", debate_file.name)

df_debate = pd.read_pickle(debate_file).copy()

print("Dimensão:", df_debate.shape)

df_debate["segment_start"] = df_debate["time stamp"]
df_debate["segment_end"] = df_debate["time stamp"] + df_debate["duration"]

df_debate.head()

embeddings = np.vstack(df_debate["speak_embeddings"].values)
embeddings_norm = normalize(embeddings)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_debate["speaker_cluster"] = kmeans.fit_predict(embeddings_norm)

speaker_time = (
    df_debate
    .groupby("speaker_cluster")
    .agg(
        n_segments=("duration", "count"),
        total_speech_sec=("duration", "sum"),
        mean_duration=("duration", "mean"),
        mean_speechrate=("speechrate", "mean")
    )
    .reset_index()
    .sort_values("total_speech_sec", ascending=False)
)

speaker_time["person"] = [f"Person {i}" for i in range(1, len(speaker_time) + 1)]

cluster_to_person = dict(zip(speaker_time["speaker_cluster"], speaker_time["person"]))

df_debate["person"] = df_debate["speaker_cluster"].map(cluster_to_person)

speaker_time["total_speech_min"] = speaker_time["total_speech_sec"] / 60
speaker_time["speech_share"] = speaker_time["total_speech_sec"] / speaker_time["total_speech_sec"].sum()

speaker_time = speaker_time.round(3)
speaker_time



In [ ]:
# grafico em função do tempo pessoa 1,2,3

person_order = ["Person 1", "Person 2", "Person 3"]

plt.figure(figsize=(14, 4))

for i, person in enumerate(person_order):
    person_segments = df_debate[df_debate["person"] == person]
    
    for _, row in person_segments.iterrows():
        plt.broken_barh(
            [(row["segment_start"], row["duration"])],
            (i - 0.4, 0.8)
        )

plt.yticks(range(len(person_order)), person_order)
plt.xlabel("Tempo do debate (segundos)")
plt.ylabel("Pessoa")
plt.title("Timeline dos segmentos de fala por pessoa — Ventura vs Seguro")
plt.tight_layout()
plt.show()

In [ ]:
# gráfico tempo de fala por pessoa

plot_data = speaker_time.sort_values("total_speech_min", ascending=True)

plt.figure(figsize=(8, 5))
bars = plt.barh(plot_data["person"], plot_data["total_speech_min"])

for bar in bars:
    width = bar.get_width()
    plt.text(
        width + 0.1,
        bar.get_y() + bar.get_height()/2,
        f"{width:.2f} min",
        va="center"
    )

plt.title("Tempo de fala estimado por pessoa — Ventura vs Seguro")
plt.xlabel("Tempo de fala estimado (minutos)")
plt.ylabel("Pessoa")
plt.tight_layout()
plt.show()

In [ ]:
# os embeddings de cada que eu usei para fazer os gráficos de cada
pca = PCA(n_components=2)
pca_result = pca.fit_transform(embeddings_norm)

df_debate["PCA1"] = pca_result[:, 0]
df_debate["PCA2"] = pca_result[:, 1]

plt.figure(figsize=(9, 7))

colors = {
    "Person 1": "tab:blue",
    "Person 2": "tab:orange",
    "Person 3": "tab:green"
}

for person in ["Person 1", "Person 2", "Person 3"]:
    data = df_debate[df_debate["person"] == person]
    
    plt.scatter(
        data["PCA1"],
        data["PCA2"],
        label=person,
        alpha=0.7,
        s=45,
        color=colors[person]
    )
    
    center_x = data["PCA1"].mean()
    center_y = data["PCA2"].mean()
    
    plt.scatter(
        center_x,
        center_y,
        color=colors[person],
        s=180,
        marker="X",
        edgecolor="black"
    )
    
    radius_x = data["PCA1"].std() * 2
    radius_y = data["PCA2"].std() * 2
    
    ellipse = Ellipse(
        (center_x, center_y),
        width=radius_x * 2,
        height=radius_y * 2,
        alpha=0.15,
        color=colors[person]
    )
    
    plt.gca().add_patch(ellipse)

plt.title("PCA dos speaker embeddings com zonas por pessoa — Ventura vs Seguro")
plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.legend(title="Speaker")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()



In [ ]:
# gráfico alterações da fala para cada pessoa

# criar blocos de tempo de 60 segundos

df_debate["time_bin"] = (df_debate["time stamp"] // 60).astype(int)

speechrate_by_bin = (
    df_debate
    .groupby(["time_bin", "person"])
    .agg(mean_speechrate=("speechrate", "mean"))
    .reset_index()
)

plt.figure(figsize=(14, 5))

for person in ["Person 1", "Person 2", "Person 3"]:
    data = speechrate_by_bin[speechrate_by_bin["person"] == person]
    plt.plot(data["time_bin"], data["mean_speechrate"], marker="o", label=person)

plt.title("Speech rate médio ao longo do tempo — Ventura vs Seguro")
plt.xlabel("Tempo (segundos)")
plt.ylabel("Speech rate médio")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# pitch médio ao longo do tempo

pitch_by_bin = (
    df_debate
    .groupby(["time_bin", "person"])
    .agg(mean_pitch=("meanF0Hz", "mean"))
    .reset_index()
)

plt.figure(figsize=(14, 5))

for person in ["Person 1", "Person 2", "Person 3"]:
    data = pitch_by_bin[pitch_by_bin["person"] == person]
    plt.plot(data["time_bin"], data["mean_pitch"], marker="o", label=person)

plt.title("Pitch médio ao longo do tempo — Ventura vs Seguro")
plt.xlabel("Tempo (segundos)")
plt.ylabel("Pitch médio (Hz)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# criar embeeding medio de cada person 

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = Path(".")
audio_files = sorted(DATA_DIR.glob("*_audio.pkl"))

def extract_metadata_from_filename(file_path):
    filename = file_path.name.replace("_audio.pkl", "")
    candidate_part, date_part = filename.split("_vs_")
    
    candidate_1 = candidate_part.replace("_", " ")
    
    date_parts = date_part.split("_")
    month = date_parts[-2]
    day = int(date_parts[-1])
    candidate_2 = " ".join(date_parts[:-2])
    
    debate_name = f"{candidate_1} vs {candidate_2}"
    
    return {
        "source_file": file_path.name,
        "debate_name": debate_name,
        "candidate_1": candidate_1,
        "candidate_2": candidate_2,
        "month": month,
        "day": day
    }

speaker_instances = []

for file_path in audio_files:
    df = pd.read_pickle(file_path).copy()
    metadata = extract_metadata_from_filename(file_path)

    embeddings = np.vstack(df["speak_embeddings"].values)
    embeddings_norm = normalize(embeddings)

    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df["local_cluster"] = kmeans.fit_predict(embeddings_norm)

    # ordenar clusters por tempo de fala
    cluster_order = (
        df.groupby("local_cluster")["duration"]
        .sum()
        .sort_values(ascending=False)
        .reset_index()
    )
    cluster_order["local_person"] = [f"Person {i}" for i in range(1, 4)]
    cluster_to_person = dict(zip(cluster_order["local_cluster"], cluster_order["local_person"]))
    df["local_person"] = df["local_cluster"].map(cluster_to_person)

    for cluster in sorted(df["local_cluster"].unique()):
        idx = df[df["local_cluster"] == cluster].index
        emb_cluster = embeddings_norm[idx]

        centroid = normalize(emb_cluster.mean(axis=0).reshape(1, -1))[0]

        row = {
            **metadata,
            "local_cluster": cluster,
            "local_person": cluster_to_person[cluster],
            "n_segments": len(idx),
            "total_speech_sec": df.loc[idx, "duration"].sum(),
            "total_speech_min": df.loc[idx, "duration"].sum() / 60,
            "mean_duration": df.loc[idx, "duration"].mean(),
            "mean_speechrate": df.loc[idx, "speechrate"].mean()
        }

        for i, value in enumerate(centroid):
            row[f"emb_{i:03d}"] = value

        speaker_instances.append(row)

speaker_instances = pd.DataFrame(speaker_instances).round(6)

print("Número de speaker instances:", speaker_instances.shape)
speaker_instances.head()


In [ ]:
#emb de cada candidato

emb_cols = [col for col in speaker_instances.columns if col.startswith("emb_")]

candidates = sorted(
    set(speaker_instances["candidate_1"]).union(set(speaker_instances["candidate_2"]))
)

candidate_embeddings = []
selected_instances_all = []

for candidate in candidates:
    candidate_debates = speaker_instances[
        (speaker_instances["candidate_1"] == candidate) |
        (speaker_instances["candidate_2"] == candidate)
    ]["debate_name"].unique()

    candidate_instances = speaker_instances[
        speaker_instances["debate_name"].isin(candidate_debates)
    ].copy()

    best_result = None

    # testar cada speaker instance como ponto de partida
    for seed_idx in candidate_instances.index:
        prototype = candidate_instances.loc[seed_idx, emb_cols].values.astype(float)
        prototype = normalize(prototype.reshape(1, -1))[0]

        for _ in range(10):
            selected_idxs = []

            for debate in candidate_debates:
                group = candidate_instances[candidate_instances["debate_name"] == debate]
                group_embeddings = group[emb_cols].values.astype(float)

                sims = group_embeddings @ prototype
                best_local_idx = group.index[np.argmax(sims)]
                selected_idxs.append(best_local_idx)

            selected_embeddings = candidate_instances.loc[selected_idxs, emb_cols].values.astype(float)
            new_prototype = normalize(selected_embeddings.mean(axis=0).reshape(1, -1))[0]

            if np.allclose(prototype, new_prototype, atol=1e-5):
                break

            prototype = new_prototype

        selected = candidate_instances.loc[selected_idxs].copy()
        selected_embeddings = selected[emb_cols].values.astype(float)
        similarities = selected_embeddings @ prototype

        mean_similarity = similarities.mean()
        min_similarity = similarities.min()

        result = {
            "candidate": candidate,
            "prototype": prototype,
            "selected": selected,
            "mean_similarity": mean_similarity,
            "min_similarity": min_similarity,
            "n_debates": len(candidate_debates)
        }

        if best_result is None or mean_similarity > best_result["mean_similarity"]:
            best_result = result

    # guardar embedding final do candidato
    row = {
        "candidate": candidate,
        "n_debates": best_result["n_debates"],
        "mean_similarity": best_result["mean_similarity"],
        "min_similarity": best_result["min_similarity"]
    }

    for i, value in enumerate(best_result["prototype"]):
        row[f"emb_{i:03d}"] = value

    candidate_embeddings.append(row)

    selected = best_result["selected"].copy()
    selected["candidate"] = candidate
    selected["similarity_to_candidate_embedding"] = (
        selected[emb_cols].values.astype(float) @ best_result["prototype"]
    )

    selected_instances_all.append(selected)

candidate_embeddings = pd.DataFrame(candidate_embeddings).round(6)
selected_instances_all = pd.concat(selected_instances_all, ignore_index=True).round(6)

candidate_embeddings[
    ["candidate", "n_debates", "mean_similarity", "min_similarity"]
].sort_values("mean_similarity", ascending=False)

In [ ]:
selected_instances_view = selected_instances_all[
    [
        "candidate",
        "debate_name",
        "candidate_1",
        "candidate_2",
        "local_person",
        "n_segments",
        "total_speech_min",
        "mean_speechrate",
        "similarity_to_candidate_embedding"
    ]
].sort_values(["candidate", "debate_name"])

selected_instances_view

In [ ]:
conflicts = (
    selected_instances_view
    .groupby(["debate_name", "local_person"])
    .agg(
        n_candidates=("candidate", "nunique"),
        candidates=("candidate", lambda x: ", ".join(sorted(x.unique())))
    )
    .reset_index()
)

conflicts = conflicts[conflicts["n_candidates"] > 1]

conflicts

In [ ]:
candidate_names = candidate_embeddings["candidate"].tolist()
candidate_matrix = candidate_embeddings[emb_cols].values.astype(float)

similarity_matrix = pd.DataFrame(
    candidate_matrix @ candidate_matrix.T,
    index=candidate_names,
    columns=candidate_names
).round(3)

similarity_matrix

In [ ]:
%pip install openpyxl
readme = pd.DataFrame({
    "Descrição": [
        "Candidate_Embeddings: embedding médio/provável de cada candidato, com 512 dimensões.",
        "Selected_Speaker_Instances: speaker escolhido automaticamente em cada debate onde o candidato participou.",
        "All_Debate_Speaker_Centroids: embeddings médios dos 3 speakers estimados em cada debate.",
        "Candidate_Similarity: matriz de similaridade entre embeddings dos candidatos.",
        "Conflicts: casos em que dois candidatos foram associados ao mesmo speaker local no mesmo debate.",
        "Nota: esta identificação é automática e deve ser validada com informação visual."
    ]
})

output_excel = "candidate_audio_embeddings.xlsx"

with pd.ExcelWriter(output_excel) as writer:
    readme.to_excel(writer, sheet_name="ReadMe", index=False)
    candidate_embeddings.to_excel(writer, sheet_name="Candidate_Embeddings", index=False)
    selected_instances_view.to_excel(writer, sheet_name="Selected_Speaker_Instances", index=False)
    speaker_instances.to_excel(writer, sheet_name="All_Debate_Speaker_Centroids", index=False)
    similarity_matrix.to_excel(writer, sheet_name="Candidate_Similarity")
    conflicts.to_excel(writer, sheet_name="Conflicts", index=False)



print("Excel guardado como:", output_excel)

In [ ]:
# guardar só os ficheiros CSV que já existem

tables_to_save = {
    "Candidate_Embeddings.csv": "candidate_embeddings",
    "Selected_Speaker_Instances.csv": "selected_instances_view",
    "All_Debate_Speaker_Centroids.csv": "speaker_instances",
    "Candidate_Similarity.csv": "similarity_matrix",
    "Conflicts.csv": "conflicts"
}

for filename, variable_name in tables_to_save.items():
    if variable_name in globals():
        globals()[variable_name].to_csv(
            filename,
            index=True if variable_name == "similarity_matrix" else False,
            sep=";",
            encoding="utf-8-sig"
        )
        print("Guardado:", filename)
    else:
        print("Não existe ainda:", variable_name)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

DATA_DIR = Path(".")
audio_files = sorted(DATA_DIR.glob("*_audio.pkl"))

print("Número de ficheiros:", len(audio_files))
def extract_metadata_from_filename(file_path):
    filename = file_path.name.replace("_audio.pkl", "")
    candidate_part, date_part = filename.split("_vs_")
    
    candidate_1 = candidate_part.replace("_", " ")
    
    date_parts = date_part.split("_")
    month = date_parts[-2]
    day = int(date_parts[-1])
    candidate_2 = " ".join(date_parts[:-2])
    
    debate_name = f"{candidate_1} vs {candidate_2}"
    
    return {
        "source_file": file_path.name,
        "debate_name": debate_name,
        "candidate_1": candidate_1,
        "candidate_2": candidate_2,
        "month": month,
        "day": day
    }
all_seconds_rows = []
all_summary_rows = []

for file_path in audio_files:
    print("A processar:", file_path.name)
    
    metadata = extract_metadata_from_filename(file_path)
    
    df = pd.read_pickle(file_path).copy()
    
    df["segment_start"] = df["time stamp"]
    df["segment_end"] = df["time stamp"] + df["duration"]
    
    # clustering de 3 vozes
    embeddings = np.vstack(df["speak_embeddings"].values)
    embeddings_norm = normalize(embeddings)
    
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df["speaker_cluster"] = kmeans.fit_predict(embeddings_norm)
    
    # ordenar clusters por tempo total de fala
    speaker_order = (
        df.groupby("speaker_cluster")["duration"]
        .sum()
        .sort_values(ascending=False)
        .reset_index()
    )
    
    speaker_order["person"] = [f"Person {i}" for i in range(1, 4)]
    cluster_to_person = dict(zip(speaker_order["speaker_cluster"], speaker_order["person"]))
    
    df["person"] = df["speaker_cluster"].map(cluster_to_person)
    
    # resumo por pessoa
    summary = (
        df.groupby("person")
        .agg(
            n_segments=("duration", "count"),
            total_speech_sec=("duration", "sum"),
            mean_segment_duration=("duration", "mean"),
            mean_speechrate=("speechrate", "mean")
        )
        .reset_index()
    )
    
    summary["total_speech_min"] = summary["total_speech_sec"] / 60
    summary["speech_share"] = summary["total_speech_sec"] / summary["total_speech_sec"].sum()
    
    for key, value in metadata.items():
        summary[key] = value
    
    all_summary_rows.append(summary)
    
    # criar uma linha por segundo
    max_second = int(np.ceil(df["segment_end"].max()))
    
    for sec in range(max_second + 1):
        sec_start = sec
        sec_end = sec + 1
        
        active = df[
            (df["segment_start"] < sec_end) &
            (df["segment_end"] > sec_start)
        ].copy()
        
        if active.empty:
            speaker = "No speech"
            overlap_seconds = 0
        else:
            active["overlap"] = (
                np.minimum(active["segment_end"], sec_end) -
                np.maximum(active["segment_start"], sec_start)
            )
            
            overlap_by_person = (
                active.groupby("person")["overlap"]
                .sum()
                .sort_values(ascending=False)
            )
            
            speaker = overlap_by_person.index[0]
            overlap_seconds = overlap_by_person.iloc[0]
        
        all_seconds_rows.append({
            "debate_name": metadata["debate_name"],
            "source_file": metadata["source_file"],
            "candidate_1": metadata["candidate_1"],
            "candidate_2": metadata["candidate_2"],
            "month": metadata["month"],
            "day": metadata["day"],
            "second": sec,
            "time_min": sec / 60,
            "estimated_speaker": speaker,
            "speaker_overlap_sec": overlap_seconds
        })

seconds_by_debate = pd.DataFrame(all_seconds_rows)
speaker_summary_all = pd.concat(all_summary_rows, ignore_index=True)

seconds_by_debate.head()




In [ ]:
output_file = "speaker_by_second_all_debates.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    seconds_by_debate.to_excel(writer, sheet_name="Second_by_second", index=False)
    speaker_summary_all.to_excel(writer, sheet_name="Speaker_summary", index=False)

print("Excel guardado como:", output_file)

In [ ]:
"all_seconds_rows" in globals()

In [ ]:
seconds_by_debate = pd.DataFrame(all_seconds_rows)

seconds_by_debate.to_csv(
    "speaker_by_second_all_debates.csv",
    index=False,
    encoding="utf-8-sig",
    sep=";"
)

print("CSV guardado como: speaker_by_second_all_debates.csv")
seconds_by_debate.head()

In [ ]:
# ficheiro csv só com os dados do debate ventura vs seguro 


from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

DATA_DIR = Path(".")
audio_files = sorted(DATA_DIR.glob("*_audio.pkl"))

# encontrar ficheiro Ventura vs Seguro
debate_file = None

for f in audio_files:
    if "Ventura" in f.name and "Seguro" in f.name:
        debate_file = f
        break

if debate_file is None:
    raise FileNotFoundError("Não encontrei o ficheiro Ventura vs Seguro.")

print("Ficheiro escolhido:", debate_file.name)

# abrir ficheiro
df = pd.read_pickle(debate_file).copy()

# criar tempos
df["segment_start"] = df["time stamp"]
df["segment_end"] = df["time stamp"] + df["duration"]

# clustering em 3 speakers
embeddings = np.vstack(df["speak_embeddings"].values)
embeddings_norm = normalize(embeddings)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["speaker_cluster"] = kmeans.fit_predict(embeddings_norm)

# ordenar clusters por tempo total de fala
speaker_order = (
    df.groupby("speaker_cluster")["duration"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

speaker_order["person"] = [f"Person {i}" for i in range(1, 4)]
cluster_to_person = dict(zip(speaker_order["speaker_cluster"], speaker_order["person"]))

df["person"] = df["speaker_cluster"].map(cluster_to_person)

# criar uma linha por segundo
rows = []

max_second = int(np.ceil(df["segment_end"].max()))

for sec in range(max_second + 1):
    sec_start = sec
    sec_end = sec + 1
    
    active = df[
        (df["segment_start"] < sec_end) &
        (df["segment_end"] > sec_start)
    ].copy()
    
    if active.empty:
        speaker = "No speech"
        overlap_seconds = 0
    else:
        active["overlap"] = (
            np.minimum(active["segment_end"], sec_end) -
            np.maximum(active["segment_start"], sec_start)
        )
        
        overlap_by_person = (
            active.groupby("person")["overlap"]
            .sum()
            .sort_values(ascending=False)
        )
        
        speaker = overlap_by_person.index[0]
        overlap_seconds = overlap_by_person.iloc[0]
    
    rows.append({
        "source_file": debate_file.name,
        "debate_name": debate_file.name.replace("_audio.pkl", "").replace("_", " "),
        "second": sec,
        "time_min": sec / 60,
        "estimated_speaker": speaker,
        "speaker_overlap_sec": overlap_seconds
    })

ventura_seguro_seconds = pd.DataFrame(rows)

ventura_seguro_seconds.to_csv(
    "ventura_seguro_speaker_by_second.csv",
    index=False,
    encoding="utf-8-sig",
    sep=";"
)

print("CSV guardado como: ventura_seguro_speaker_by_second.csv")
ventura_seguro_seconds.head(20)

In [ ]:
# agora com a informaçao do csv tirar a pessoa já com o nome para ser usado no visual 

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = Path(".")
audio_files = sorted(DATA_DIR.glob("*_audio.pkl"))

def extract_metadata_from_filename(file_path):
    filename = file_path.name.replace("_audio.pkl", "")
    candidate_part, date_part = filename.split("_vs_")
    
    candidate_1 = candidate_part.replace("_", " ")
    
    date_parts = date_part.split("_")
    month = date_parts[-2]
    day = int(date_parts[-1])
    candidate_2 = " ".join(date_parts[:-2])
    
    debate_name = f"{candidate_1} vs {candidate_2}"
    
    return {
        "source_file": file_path.name,
        "debate_name": debate_name,
        "candidate_1": candidate_1,
        "candidate_2": candidate_2,
        "month": month,
        "day": day
    }

# 1. Criar centroids dos 3 speakers em cada debate

speaker_instances = []

for file_path in audio_files:
    df = pd.read_pickle(file_path).copy()
    metadata = extract_metadata_from_filename(file_path)
    
    embeddings = np.vstack(df["speak_embeddings"].values)
    embeddings_norm = normalize(embeddings)
    
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df["speaker_cluster"] = kmeans.fit_predict(embeddings_norm)
    
    for cluster in sorted(df["speaker_cluster"].unique()):
        idx = df[df["speaker_cluster"] == cluster].index
        centroid = normalize(embeddings_norm[idx].mean(axis=0).reshape(1, -1))[0]
        
        row = {
            **metadata,
            "speaker_cluster": cluster,
            "total_speech_sec": df.loc[idx, "duration"].sum(),
            "n_segments": len(idx)
        }
        
        for i, value in enumerate(centroid):
            row[f"emb_{i:03d}"] = value
        
        speaker_instances.append(row)

speaker_instances = pd.DataFrame(speaker_instances)

emb_cols = [c for c in speaker_instances.columns if c.startswith("emb_")]

# 2. Função para encontrar o embedding provável de um candidato
# usando os debates onde ele aparece

def infer_candidate_embedding(candidate, exclude_source_file=None):
    candidate_debates = speaker_instances[
        (speaker_instances["candidate_1"] == candidate) |
        (speaker_instances["candidate_2"] == candidate)
    ].copy()
    
    if exclude_source_file is not None:
        candidate_debates = candidate_debates[
            candidate_debates["source_file"] != exclude_source_file
        ]
    
    debate_names = candidate_debates["debate_name"].unique()
    
    best_score = -999
    best_prototype = None
    
    for seed_idx in candidate_debates.index:
        prototype = candidate_debates.loc[seed_idx, emb_cols].values.astype(float)
        prototype = normalize(prototype.reshape(1, -1))[0]
        
        for _ in range(10):
            selected_embeddings = []
            
            for debate in debate_names:
                group = candidate_debates[candidate_debates["debate_name"] == debate]
                group_embs = group[emb_cols].values.astype(float)
                
                sims = group_embs @ prototype
                best_local = group_embs[np.argmax(sims)]
                selected_embeddings.append(best_local)
            
            selected_embeddings = np.vstack(selected_embeddings)
            new_prototype = normalize(selected_embeddings.mean(axis=0).reshape(1, -1))[0]
            
            if np.allclose(prototype, new_prototype, atol=1e-5):
                break
            
            prototype = new_prototype
        
        sims_final = selected_embeddings @ prototype
        score = sims_final.mean()
        
        if score > best_score:
            best_score = score
            best_prototype = prototype
    
    return best_prototype, best_score

# 3. Encontrar o ficheiro Ventura vs Seguro

target_file = None

for f in audio_files:
    if "Ventura" in f.name and "Seguro" in f.name:
        target_file = f
        break

if target_file is None:
    raise FileNotFoundError("Não encontrei o ficheiro Ventura vs Seguro.")

print("Ficheiro escolhido:", target_file.name)

# 4. Criar embeddings prováveis de Ventura e Seguro,
# excluindo o próprio debate Ventura vs Seguro

ventura_emb, ventura_score = infer_candidate_embedding(
    "Ventura",
    exclude_source_file=target_file.name
)

seguro_emb, seguro_score = infer_candidate_embedding(
    "Seguro",
    exclude_source_file=target_file.name
)

print("Score Ventura:", round(ventura_score, 3))
print("Score Seguro:", round(seguro_score, 3))

# 5. Abrir o debate Ventura vs Seguro e clusterizar em 3 speakers

df_target = pd.read_pickle(target_file).copy()

df_target["segment_start"] = df_target["time stamp"]
df_target["segment_end"] = df_target["time stamp"] + df_target["duration"]

embeddings = np.vstack(df_target["speak_embeddings"].values)
embeddings_norm = normalize(embeddings)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_target["speaker_cluster"] = kmeans.fit_predict(embeddings_norm)

# 6. Criar centroid de cada cluster no debate escolhido

target_centroids = {}

for cluster in sorted(df_target["speaker_cluster"].unique()):
    idx = df_target[df_target["speaker_cluster"] == cluster].index
    centroid = normalize(embeddings_norm[idx].mean(axis=0).reshape(1, -1))[0]
    target_centroids[cluster] = centroid

# 7. Atribuir clusters a Ventura, Seguro e Moderador

clusters = list(target_centroids.keys())

best_pair = None
best_score = -999

for c_ventura in clusters:
    for c_seguro in clusters:
        if c_ventura == c_seguro:
            continue
        
        sim_ventura = np.dot(target_centroids[c_ventura], ventura_emb)
        sim_seguro = np.dot(target_centroids[c_seguro], seguro_emb)
        
        score = sim_ventura + sim_seguro
        
        if score > best_score:
            best_score = score
            best_pair = (c_ventura, c_seguro, sim_ventura, sim_seguro)

ventura_cluster, seguro_cluster, sim_ventura, sim_seguro = best_pair

cluster_to_name = {}

for cluster in clusters:
    if cluster == ventura_cluster:
        cluster_to_name[cluster] = "Ventura"
    elif cluster == seguro_cluster:
        cluster_to_name[cluster] = "Seguro"
    else:
        cluster_to_name[cluster] = "Moderador"

df_target["speaker_name"] = df_target["speaker_cluster"].map(cluster_to_name)

print("Mapeamento encontrado:")
print(cluster_to_name)
print("Similaridade Ventura:", round(sim_ventura, 3))
print("Similaridade Seguro:", round(sim_seguro, 3))

# 8. Criar uma linha por segundo

rows = []

max_second = int(np.ceil(df_target["segment_end"].max()))

for sec in range(max_second + 1):
    sec_start = sec
    sec_end = sec + 1
    
    active = df_target[
        (df_target["segment_start"] < sec_end) &
        (df_target["segment_end"] > sec_start)
    ].copy()
    
    if active.empty:
        speaker = "No speech"
        overlap_seconds = 0
    else:
        active["overlap"] = (
            np.minimum(active["segment_end"], sec_end) -
            np.maximum(active["segment_start"], sec_start)
        )
        
        overlap_by_person = (
            active.groupby("speaker_name")["overlap"]
            .sum()
            .sort_values(ascending=False)
        )
        
        speaker = overlap_by_person.index[0]
        overlap_seconds = overlap_by_person.iloc[0]
    
    rows.append({
        "source_file": target_file.name,
        "debate_name": "Ventura vs Seguro",
        "second": sec,
        "time_min": sec / 60,
        "speaker_name": speaker,
        "speaker_overlap_sec": overlap_seconds
    })

ventura_seguro_seconds_named = pd.DataFrame(rows)

# 9. Guardar CSV

ventura_seguro_seconds_named.to_csv(
    "ventura_seguro_speaker_by_second_named.csv",
    index=False,
    encoding="utf-8-sig",
    sep=";"
)

print("CSV guardado como: ventura_seguro_speaker_by_second_named.csv")

ventura_seguro_seconds_named.head(20)

In [ ]:
## agora para todos os debates ---- meu deus 

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

DATA_DIR = Path(".")
audio_files = sorted(DATA_DIR.glob("*_audio.pkl"))

def extract_metadata_from_filename(file_path):
    filename = file_path.name.replace("_audio.pkl", "")
    candidate_part, date_part = filename.split("_vs_")
    
    candidate_1 = candidate_part.replace("_", " ")
    
    date_parts = date_part.split("_")
    month = date_parts[-2]
    day = int(date_parts[-1])
    candidate_2 = " ".join(date_parts[:-2])
    
    debate_name = f"{candidate_1} vs {candidate_2}"
    
    return {
        "source_file": file_path.name,
        "debate_name": debate_name,
        "candidate_1": candidate_1,
        "candidate_2": candidate_2,
        "month": month,
        "day": day
    }




In [ ]:
# clusterssssss

speaker_instances = []
debate_segment_data = {}

for file_path in audio_files:
    print("A processar:", file_path.name)
    
    metadata = extract_metadata_from_filename(file_path)
    
    df = pd.read_pickle(file_path).copy()
    df["segment_start"] = df["time stamp"]
    df["segment_end"] = df["time stamp"] + df["duration"]
    
    embeddings = np.vstack(df["speak_embeddings"].values)
    embeddings_norm = normalize(embeddings)
    
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df["speaker_cluster"] = kmeans.fit_predict(embeddings_norm)
    
    debate_segment_data[file_path.name] = {
        "df": df,
        "metadata": metadata,
        "embeddings_norm": embeddings_norm
    }
    
    for cluster in sorted(df["speaker_cluster"].unique()):
        idx = df[df["speaker_cluster"] == cluster].index
        centroid = normalize(embeddings_norm[idx].mean(axis=0).reshape(1, -1))[0]
        
        row = {
            **metadata,
            "speaker_cluster": cluster,
            "total_speech_sec": df.loc[idx, "duration"].sum(),
            "n_segments": len(idx)
        }
        
        for i, value in enumerate(centroid):
            row[f"emb_{i:03d}"] = value
        
        speaker_instances.append(row)

speaker_instances = pd.DataFrame(speaker_instances)

emb_cols = [c for c in speaker_instances.columns if c.startswith("emb_")]

speaker_instances.head()

In [ ]:
candidates = sorted(
    set(speaker_instances["candidate_1"]).union(set(speaker_instances["candidate_2"]))
)

def infer_candidate_embedding(candidate):
    candidate_instances = speaker_instances[
        (speaker_instances["candidate_1"] == candidate) |
        (speaker_instances["candidate_2"] == candidate)
    ].copy()
    
    debate_names = candidate_instances["debate_name"].unique()
    
    best_score = -999
    best_prototype = None
    
    for seed_idx in candidate_instances.index:
        prototype = candidate_instances.loc[seed_idx, emb_cols].values.astype(float)
        prototype = normalize(prototype.reshape(1, -1))[0]
        
        for _ in range(10):
            selected_embeddings = []
            
            for debate in debate_names:
                group = candidate_instances[candidate_instances["debate_name"] == debate]
                group_embs = group[emb_cols].values.astype(float)
                
                sims = group_embs @ prototype
                best_local = group_embs[np.argmax(sims)]
                selected_embeddings.append(best_local)
            
            selected_embeddings = np.vstack(selected_embeddings)
            new_prototype = normalize(selected_embeddings.mean(axis=0).reshape(1, -1))[0]
            
            if np.allclose(prototype, new_prototype, atol=1e-5):
                break
            
            prototype = new_prototype
        
        sims_final = selected_embeddings @ prototype
        score = sims_final.mean()
        
        if score > best_score:
            best_score = score
            best_prototype = prototype
    
    return best_prototype, best_score


candidate_embeddings = {}

for candidate in candidates:
    emb, score = infer_candidate_embedding(candidate)
    candidate_embeddings[candidate] = {
        "embedding": emb,
        "score": score
    }

candidate_embedding_quality = pd.DataFrame([
    {
        "candidate": candidate,
        "embedding_score": values["score"]
    }
    for candidate, values in candidate_embeddings.items()
]).round(3)

candidate_embedding_quality

In [ ]:
mapping_rows = []
all_seconds_rows = []
all_speaker_summary_rows = []

for source_file, data in debate_segment_data.items():
    df = data["df"].copy()
    metadata = data["metadata"]
    embeddings_norm = data["embeddings_norm"]
    
    candidate_1 = metadata["candidate_1"]
    candidate_2 = metadata["candidate_2"]
    
    clusters = sorted(df["speaker_cluster"].unique())
    
    cluster_centroids = {}
    
    for cluster in clusters:
        idx = df[df["speaker_cluster"] == cluster].index
        centroid = normalize(embeddings_norm[idx].mean(axis=0).reshape(1, -1))[0]
        cluster_centroids[cluster] = centroid
    
    emb_1 = candidate_embeddings[candidate_1]["embedding"]
    emb_2 = candidate_embeddings[candidate_2]["embedding"]
    
    best_score = -999
    best_pair = None
    
    for c1 in clusters:
        for c2 in clusters:
            if c1 == c2:
                continue
            
            sim_1 = np.dot(cluster_centroids[c1], emb_1)
            sim_2 = np.dot(cluster_centroids[c2], emb_2)
            score = sim_1 + sim_2
            
            if score > best_score:
                best_score = score
                best_pair = (c1, c2, sim_1, sim_2)
    
    cluster_candidate_1, cluster_candidate_2, sim_1, sim_2 = best_pair
    
    cluster_to_name = {}
    
    for cluster in clusters:
        if cluster == cluster_candidate_1:
            cluster_to_name[cluster] = candidate_1
        elif cluster == cluster_candidate_2:
            cluster_to_name[cluster] = candidate_2
        else:
            cluster_to_name[cluster] = "Moderador/Other"
    
    df["speaker_name"] = df["speaker_cluster"].map(cluster_to_name)
    
    mapping_rows.append({
        **metadata,
        "cluster_candidate_1": cluster_candidate_1,
        "cluster_candidate_2": cluster_candidate_2,
        "sim_candidate_1": sim_1,
        "sim_candidate_2": sim_2,
        "mapping_score": best_score,
        "cluster_to_name": str(cluster_to_name)
    })
    
    speaker_summary = (
        df.groupby("speaker_name")
        .agg(
            n_segments=("duration", "count"),
            total_speech_sec=("duration", "sum"),
            mean_segment_duration=("duration", "mean"),
            mean_speechrate=("speechrate", "mean")
        )
        .reset_index()
    )
    
    speaker_summary["total_speech_min"] = speaker_summary["total_speech_sec"] / 60
    speaker_summary["speech_share"] = (
        speaker_summary["total_speech_sec"] / speaker_summary["total_speech_sec"].sum()
    )
    
    for key, value in metadata.items():
        speaker_summary[key] = value
    
    all_speaker_summary_rows.append(speaker_summary)
    
    max_second = int(np.ceil(df["segment_end"].max()))
    
    for sec in range(max_second + 1):
        sec_start = sec
        sec_end = sec + 1
        
        active = df[
            (df["segment_start"] < sec_end) &
            (df["segment_end"] > sec_start)
        ].copy()
        
        if active.empty:
            speaker = "No speech"
            overlap_seconds = 0
        else:
            active["overlap"] = (
                np.minimum(active["segment_end"], sec_end) -
                np.maximum(active["segment_start"], sec_start)
            )
            
            overlap_by_person = (
                active.groupby("speaker_name")["overlap"]
                .sum()
                .sort_values(ascending=False)
            )
            
            speaker = overlap_by_person.index[0]
            overlap_seconds = overlap_by_person.iloc[0]
        
        all_seconds_rows.append({
            **metadata,
            "second": sec,
            "time_min": sec / 60,
            "speaker_name": speaker,
            "speaker_overlap_sec": overlap_seconds
        })

speaker_mapping_all = pd.DataFrame(mapping_rows).round(3)
speaker_summary_named_all = pd.concat(all_speaker_summary_rows, ignore_index=True).round(3)
seconds_named_all_debates = pd.DataFrame(all_seconds_rows)

speaker_mapping_all.head()

In [ ]:
seconds_named_all_debates.to_csv(
    "speaker_by_second_all_debates_named.csv",
    index=False,
    encoding="utf-8-sig",
    sep=";"
)

speaker_summary_named_all.to_csv(
    "speaker_summary_all_debates_named.csv",
    index=False,
    encoding="utf-8-sig",
    sep=";"
)

speaker_mapping_all.to_csv(
    "speaker_mapping_all_debates_named.csv",
    index=False,
    encoding="utf-8-sig",
    sep=";"
)

candidate_embedding_quality.to_csv(
    "candidate_embedding_quality.csv",
    index=False,
    encoding="utf-8-sig",
    sep=";"
)

print("Ficheiros guardados:")
print("- speaker_by_second_all_debates_named.csv")
print("- speaker_summary_all_debates_named.csv")
print("- speaker_mapping_all_debates_named.csv")
print("- candidate_embedding_quality.csv")